In [22]:
import pandas as pd
import numpy as np
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
data=pd.read_pickle(r"C:\Users\user\Downloads\Projet ML\Model1(KNN)\Données nettoyées\fusion_final.pkl.gz")
data

,tconst,titleType,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,French_Title,averageRating,numVotes,primaryName
0,tt0003883,movie,L'enfant de Paris (1913),L'enfant de Paris (1913),1913,124.0,"Crime,Drama",L'Enfant de Paris (1913),7.2,446.0,"Léonce Perret,Louis Leubas,Maurice Lagrenée,Ém..."
1,tt0006206,movie,Les vampires (1915),Les vampires (1915),1915,421.0,"Action,Adventure,Crime",Les vampires (1915),7.3,5398.0,"Stacia Napierkowska,Musidora,Édouard Mathé,Mar..."
2,tt0006864,movie,Intolerance (1916),Intolerance: Love's Struggle Throughout the Ag...,1916,163.0,"Drama,History",Intolérance (1916),7.7,16642.0,"Lillian Gish,Robert Harron,Mae Marsh,F.A. Turner"
3,tt0006886,movie,Judex (1916),Judex (1916),1916,300.0,"Adventure,Crime",Judex (1916),7.2,1117.0,"René Cresté,Musidora,René Poyen,Édouard Mathé"
4,tt0009968,movie,Broken Blossoms (1919),Broken Blossoms or The Yellow Man and the Girl...,1919,90.0,"Drama,Romance",Le lys brisé (1919),7.2,11014.0,"Norman Selby,Lillian Gish,Richard Barthelmess,..."
...,...,...,...,...,...,...,...,...,...,...,...
13089,tt9900782,movie,Kaithi (2019),Kaithi (2019),2019,145.0,"Action,Crime,Drama",Kaithi (2019),8.4,42096.0,"Karthi,Narain,Arjun Das,George Maryan"
13090,tt9904844,movie,Ott Tänak: The Movie (2019),Ott Tänak: The Movie (2019),2019,125.0,"Documentary,Sport",NaN,8.1,501.0,NaN
13091,tt9905462,movie,Pengalila (2019),Pengalila (2019),2019,111.0,Drama,NaN,7.6,680.0,NaN
13092,tt9911196,movie,The Marriage Escape (2020),De beentjes van Sint-Hildegard (2020),2020,103.0,"Comedy,Drama",De Beentjes van Sint-Hildegard (2020),7.4,3306.0,"Herman Finkers,Johanna ter Steege,Leonie ter B..."


In [24]:
data.head()

,tconst,titleType,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,French_Title,averageRating,numVotes,primaryName
0,tt0003883,movie,L'enfant de Paris (1913),L'enfant de Paris (1913),1913,124.0,"Crime,Drama",L'Enfant de Paris (1913),7.2,446.0,"Léonce Perret,Louis Leubas,Maurice Lagrenée,Ém..."
1,tt0006206,movie,Les vampires (1915),Les vampires (1915),1915,421.0,"Action,Adventure,Crime",Les vampires (1915),7.3,5398.0,"Stacia Napierkowska,Musidora,Édouard Mathé,Mar..."
2,tt0006864,movie,Intolerance (1916),Intolerance: Love's Struggle Throughout the Ag...,1916,163.0,"Drama,History",Intolérance (1916),7.7,16642.0,"Lillian Gish,Robert Harron,Mae Marsh,F.A. Turner"
3,tt0006886,movie,Judex (1916),Judex (1916),1916,300.0,"Adventure,Crime",Judex (1916),7.2,1117.0,"René Cresté,Musidora,René Poyen,Édouard Mathé"
4,tt0009968,movie,Broken Blossoms (1919),Broken Blossoms or The Yellow Man and the Girl...,1919,90.0,"Drama,Romance",Le lys brisé (1919),7.2,11014.0,"Norman Selby,Lillian Gish,Richard Barthelmess,..."


In [25]:
data.isnull().sum()

tconst               0
titleType            0
primaryTitle         0
originalTitle        0
startYear            0
runtimeMinutes       0
genres               0
French_Title      5571
averageRating        0
numVotes             0
primaryName       2174
dtype: int64

In [26]:
#Nombre de valeurs en double
data['tconst'].duplicated().sum() 

772

In [27]:
#Supprimer ces valeurs en double
data = data.drop_duplicates(subset = "tconst")

In [28]:
# Réindexer le DataFrame après la suppression des doublons
data = data.reset_index(drop=True)

In [29]:
data

,tconst,titleType,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,French_Title,averageRating,numVotes,primaryName
0,tt0003883,movie,L'enfant de Paris (1913),L'enfant de Paris (1913),1913,124.0,"Crime,Drama",L'Enfant de Paris (1913),7.2,446.0,"Léonce Perret,Louis Leubas,Maurice Lagrenée,Ém..."
1,tt0006206,movie,Les vampires (1915),Les vampires (1915),1915,421.0,"Action,Adventure,Crime",Les vampires (1915),7.3,5398.0,"Stacia Napierkowska,Musidora,Édouard Mathé,Mar..."
2,tt0006864,movie,Intolerance (1916),Intolerance: Love's Struggle Throughout the Ag...,1916,163.0,"Drama,History",Intolérance (1916),7.7,16642.0,"Lillian Gish,Robert Harron,Mae Marsh,F.A. Turner"
3,tt0006886,movie,Judex (1916),Judex (1916),1916,300.0,"Adventure,Crime",Judex (1916),7.2,1117.0,"René Cresté,Musidora,René Poyen,Édouard Mathé"
4,tt0009968,movie,Broken Blossoms (1919),Broken Blossoms or The Yellow Man and the Girl...,1919,90.0,"Drama,Romance",Le lys brisé (1919),7.2,11014.0,"Norman Selby,Lillian Gish,Richard Barthelmess,..."
...,...,...,...,...,...,...,...,...,...,...,...
12317,tt9900782,movie,Kaithi (2019),Kaithi (2019),2019,145.0,"Action,Crime,Drama",Kaithi (2019),8.4,42096.0,"Karthi,Narain,Arjun Das,George Maryan"
12318,tt9904844,movie,Ott Tänak: The Movie (2019),Ott Tänak: The Movie (2019),2019,125.0,"Documentary,Sport",NaN,8.1,501.0,NaN
12319,tt9905462,movie,Pengalila (2019),Pengalila (2019),2019,111.0,Drama,NaN,7.6,680.0,NaN
12320,tt9911196,movie,The Marriage Escape (2020),De beentjes van Sint-Hildegard (2020),2020,103.0,"Comedy,Drama",De Beentjes van Sint-Hildegard (2020),7.4,3306.0,"Herman Finkers,Johanna ter Steege,Leonie ter B..."


In [30]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12322 entries, 0 to 12321
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tconst          12322 non-null  object 
 1   titleType       12322 non-null  object 
 2   primaryTitle    12322 non-null  object 
 3   originalTitle   12322 non-null  object 
 4   startYear       12322 non-null  int64  
 5   runtimeMinutes  12322 non-null  float64
 6   genres          12322 non-null  object 
 7   French_Title    6751 non-null   object 
 8   averageRating   12322 non-null  float64
 9   numVotes        12322 non-null  float64
 10  primaryName     10195 non-null  object 
dtypes: float64(3), int64(1), object(7)
memory usage: 1.0+ MB


In [31]:
#Remove space
def remove_space(text):
    if isinstance(text, str):  # Vérifier si le texte est une chaîne de caractères
        return text.replace(" ", "")
    else:
        return text  # Retourner la valeur sans modification si ce n'est pas une chaîne de caractères

In [32]:
data['primaryName']=data['primaryName'].apply(remove_space)
data['genres']=data['genres'].apply(remove_space)

In [33]:
data

,tconst,titleType,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,French_Title,averageRating,numVotes,primaryName
0,tt0003883,movie,L'enfant de Paris (1913),L'enfant de Paris (1913),1913,124.0,"Crime,Drama",L'Enfant de Paris (1913),7.2,446.0,"LéoncePerret,LouisLeubas,MauriceLagrenée,Émile..."
1,tt0006206,movie,Les vampires (1915),Les vampires (1915),1915,421.0,"Action,Adventure,Crime",Les vampires (1915),7.3,5398.0,"StaciaNapierkowska,Musidora,ÉdouardMathé,Marce..."
2,tt0006864,movie,Intolerance (1916),Intolerance: Love's Struggle Throughout the Ag...,1916,163.0,"Drama,History",Intolérance (1916),7.7,16642.0,"LillianGish,RobertHarron,MaeMarsh,F.A.Turner"
3,tt0006886,movie,Judex (1916),Judex (1916),1916,300.0,"Adventure,Crime",Judex (1916),7.2,1117.0,"RenéCresté,Musidora,RenéPoyen,ÉdouardMathé"
4,tt0009968,movie,Broken Blossoms (1919),Broken Blossoms or The Yellow Man and the Girl...,1919,90.0,"Drama,Romance",Le lys brisé (1919),7.2,11014.0,"NormanSelby,LillianGish,RichardBarthelmess,Don..."
...,...,...,...,...,...,...,...,...,...,...,...
12317,tt9900782,movie,Kaithi (2019),Kaithi (2019),2019,145.0,"Action,Crime,Drama",Kaithi (2019),8.4,42096.0,"Karthi,Narain,ArjunDas,GeorgeMaryan"
12318,tt9904844,movie,Ott Tänak: The Movie (2019),Ott Tänak: The Movie (2019),2019,125.0,"Documentary,Sport",NaN,8.1,501.0,NaN
12319,tt9905462,movie,Pengalila (2019),Pengalila (2019),2019,111.0,Drama,NaN,7.6,680.0,NaN
12320,tt9911196,movie,The Marriage Escape (2020),De beentjes van Sint-Hildegard (2020),2020,103.0,"Comedy,Drama",De Beentjes van Sint-Hildegard (2020),7.4,3306.0,"HermanFinkers,JohannaterSteege,LeonieterBraak,..."


In [34]:
data['genres'] = data['genres'].str.replace(',', ' ')
data['primaryName'] = data['primaryName'].str.replace(',', ' ')
data['runtimeMinutes'] = data['runtimeMinutes'].astype(int)
data['runtimeMinutes'] = data['runtimeMinutes'].astype(str)
data

,tconst,titleType,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,French_Title,averageRating,numVotes,primaryName
0,tt0003883,movie,L'enfant de Paris (1913),L'enfant de Paris (1913),1913,124,Crime Drama,L'Enfant de Paris (1913),7.2,446.0,LéoncePerret LouisLeubas MauriceLagrenée Émile...
1,tt0006206,movie,Les vampires (1915),Les vampires (1915),1915,421,Action Adventure Crime,Les vampires (1915),7.3,5398.0,StaciaNapierkowska Musidora ÉdouardMathé Marce...
2,tt0006864,movie,Intolerance (1916),Intolerance: Love's Struggle Throughout the Ag...,1916,163,Drama History,Intolérance (1916),7.7,16642.0,LillianGish RobertHarron MaeMarsh F.A.Turner
3,tt0006886,movie,Judex (1916),Judex (1916),1916,300,Adventure Crime,Judex (1916),7.2,1117.0,RenéCresté Musidora RenéPoyen ÉdouardMathé
4,tt0009968,movie,Broken Blossoms (1919),Broken Blossoms or The Yellow Man and the Girl...,1919,90,Drama Romance,Le lys brisé (1919),7.2,11014.0,NormanSelby LillianGish RichardBarthelmess Don...
...,...,...,...,...,...,...,...,...,...,...,...
12317,tt9900782,movie,Kaithi (2019),Kaithi (2019),2019,145,Action Crime Drama,Kaithi (2019),8.4,42096.0,Karthi Narain ArjunDas GeorgeMaryan
12318,tt9904844,movie,Ott Tänak: The Movie (2019),Ott Tänak: The Movie (2019),2019,125,Documentary Sport,NaN,8.1,501.0,NaN
12319,tt9905462,movie,Pengalila (2019),Pengalila (2019),2019,111,Drama,NaN,7.6,680.0,NaN
12320,tt9911196,movie,The Marriage Escape (2020),De beentjes van Sint-Hildegard (2020),2020,103,Comedy Drama,De Beentjes van Sint-Hildegard (2020),7.4,3306.0,HermanFinkers JohannaterSteege LeonieterBraak ...


In [35]:
# Convertir  les colonnes en chaînes de caractères
data['genres'] = data['genres'].astype(str)
data['primaryName'] = data['primaryName'].astype(str)
data['runtimeMinutes'] = data['runtimeMinutes'] + 'min'
data['startYear']=data['startYear'].astype(str)
# Appliquer la transformation pour créer la colonne 'tags'
data['tags'] = data.apply(lambda row: ' '.join(row['genres'].split(',')) + ' ' + ' '.join(row['primaryName'].split(',')) + ' ' + row['runtimeMinutes'] + ' ' + row['startYear'], axis=1)

In [36]:
data

,tconst,titleType,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,French_Title,averageRating,numVotes,primaryName,tags
0,tt0003883,movie,L'enfant de Paris (1913),L'enfant de Paris (1913),1913,124min,Crime Drama,L'Enfant de Paris (1913),7.2,446.0,LéoncePerret LouisLeubas MauriceLagrenée Émile...,Crime Drama LéoncePerret LouisLeubas MauriceLa...
1,tt0006206,movie,Les vampires (1915),Les vampires (1915),1915,421min,Action Adventure Crime,Les vampires (1915),7.3,5398.0,StaciaNapierkowska Musidora ÉdouardMathé Marce...,Action Adventure Crime StaciaNapierkowska Musi...
2,tt0006864,movie,Intolerance (1916),Intolerance: Love's Struggle Throughout the Ag...,1916,163min,Drama History,Intolérance (1916),7.7,16642.0,LillianGish RobertHarron MaeMarsh F.A.Turner,Drama History LillianGish RobertHarron MaeMars...
3,tt0006886,movie,Judex (1916),Judex (1916),1916,300min,Adventure Crime,Judex (1916),7.2,1117.0,RenéCresté Musidora RenéPoyen ÉdouardMathé,Adventure Crime RenéCresté Musidora RenéPoyen ...
4,tt0009968,movie,Broken Blossoms (1919),Broken Blossoms or The Yellow Man and the Girl...,1919,90min,Drama Romance,Le lys brisé (1919),7.2,11014.0,NormanSelby LillianGish RichardBarthelmess Don...,Drama Romance NormanSelby LillianGish RichardB...
...,...,...,...,...,...,...,...,...,...,...,...,...
12317,tt9900782,movie,Kaithi (2019),Kaithi (2019),2019,145min,Action Crime Drama,Kaithi (2019),8.4,42096.0,Karthi Narain ArjunDas GeorgeMaryan,Action Crime Drama Karthi Narain ArjunDas Geor...
12318,tt9904844,movie,Ott Tänak: The Movie (2019),Ott Tänak: The Movie (2019),2019,125min,Documentary Sport,NaN,8.1,501.0,nan,Documentary Sport nan 125min 2019
12319,tt9905462,movie,Pengalila (2019),Pengalila (2019),2019,111min,Drama,NaN,7.6,680.0,nan,Drama nan 111min 2019
12320,tt9911196,movie,The Marriage Escape (2020),De beentjes van Sint-Hildegard (2020),2020,103min,Comedy Drama,De Beentjes van Sint-Hildegard (2020),7.4,3306.0,HermanFinkers JohannaterSteege LeonieterBraak ...,Comedy Drama HermanFinkers JohannaterSteege Le...


In [37]:
data.iloc[8]['tags']

'Drama Fantasy Horror EinarAxelsson VictorSjöström HildaBorgström ToreSvennberg AstridHolm ConcordiaSelander LisaLundholm TorWeijden 107min 1921'

In [38]:
new_data=data[['tconst','primaryTitle','originalTitle','French_Title','tags']]
new_data

,tconst,primaryTitle,originalTitle,French_Title,tags
0,tt0003883,L'enfant de Paris (1913),L'enfant de Paris (1913),L'Enfant de Paris (1913),Crime Drama LéoncePerret LouisLeubas MauriceLa...
1,tt0006206,Les vampires (1915),Les vampires (1915),Les vampires (1915),Action Adventure Crime StaciaNapierkowska Musi...
2,tt0006864,Intolerance (1916),Intolerance: Love's Struggle Throughout the Ag...,Intolérance (1916),Drama History LillianGish RobertHarron MaeMars...
3,tt0006886,Judex (1916),Judex (1916),Judex (1916),Adventure Crime RenéCresté Musidora RenéPoyen ...
4,tt0009968,Broken Blossoms (1919),Broken Blossoms or The Yellow Man and the Girl...,Le lys brisé (1919),Drama Romance NormanSelby LillianGish RichardB...
...,...,...,...,...,...
12317,tt9900782,Kaithi (2019),Kaithi (2019),Kaithi (2019),Action Crime Drama Karthi Narain ArjunDas Geor...
12318,tt9904844,Ott Tänak: The Movie (2019),Ott Tänak: The Movie (2019),NaN,Documentary Sport nan 125min 2019
12319,tt9905462,Pengalila (2019),Pengalila (2019),NaN,Drama nan 111min 2019
12320,tt9911196,The Marriage Escape (2020),De beentjes van Sint-Hildegard (2020),De Beentjes van Sint-Hildegard (2020),Comedy Drama HermanFinkers JohannaterSteege Le...


In [39]:
#Transformer tags en lettre minuscules
new_data['tags']=new_data['tags'].apply(lambda x :x.lower())
new_data

C:\Users\user\AppData\Local\Temp\ipykernel_16132\3890256331.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['tags']=new_data['tags'].apply(lambda x :x.lower())


,tconst,primaryTitle,originalTitle,French_Title,tags
0,tt0003883,L'enfant de Paris (1913),L'enfant de Paris (1913),L'Enfant de Paris (1913),crime drama léonceperret louisleubas mauricela...
1,tt0006206,Les vampires (1915),Les vampires (1915),Les vampires (1915),action adventure crime stacianapierkowska musi...
2,tt0006864,Intolerance (1916),Intolerance: Love's Struggle Throughout the Ag...,Intolérance (1916),drama history lilliangish robertharron maemars...
3,tt0006886,Judex (1916),Judex (1916),Judex (1916),adventure crime renécresté musidora renépoyen ...
4,tt0009968,Broken Blossoms (1919),Broken Blossoms or The Yellow Man and the Girl...,Le lys brisé (1919),drama romance normanselby lilliangish richardb...
...,...,...,...,...,...
12317,tt9900782,Kaithi (2019),Kaithi (2019),Kaithi (2019),action crime drama karthi narain arjundas geor...
12318,tt9904844,Ott Tänak: The Movie (2019),Ott Tänak: The Movie (2019),NaN,documentary sport nan 125min 2019
12319,tt9905462,Pengalila (2019),Pengalila (2019),NaN,drama nan 111min 2019
12320,tt9911196,The Marriage Escape (2020),De beentjes van Sint-Hildegard (2020),De Beentjes van Sint-Hildegard (2020),comedy drama hermanfinkers johannatersteege le...


In [40]:
new_data.head()

,tconst,primaryTitle,originalTitle,French_Title,tags
0,tt0003883,L'enfant de Paris (1913),L'enfant de Paris (1913),L'Enfant de Paris (1913),crime drama léonceperret louisleubas mauricela...
1,tt0006206,Les vampires (1915),Les vampires (1915),Les vampires (1915),action adventure crime stacianapierkowska musi...
2,tt0006864,Intolerance (1916),Intolerance: Love's Struggle Throughout the Ag...,Intolérance (1916),drama history lilliangish robertharron maemars...
3,tt0006886,Judex (1916),Judex (1916),Judex (1916),adventure crime renécresté musidora renépoyen ...
4,tt0009968,Broken Blossoms (1919),Broken Blossoms or The Yellow Man and the Girl...,Le lys brisé (1919),drama romance normanselby lilliangish richardb...


In [41]:
new_data.to_csv(r"C:\Users\user\Downloads\Projet ML\Model2 textvectorclassificator(countvectorizer)\Dataset1\new_data.csv")

In [42]:
new_data.iloc[8]['tags']

'drama fantasy horror einaraxelsson victorsjöström hildaborgström toresvennberg astridholm concordiaselander lisalundholm torweijden 107min 1921'

# Racinisation (ou stemming) des mots dans un texte donné¶

In [43]:
# Avec l'algorithme Snowball
sb = SnowballStemmer(language='english') # initialisation avec la langue anglaise

In [44]:
def stem(text):
    L = []
    for i in text.split():
        L.append(sb.stem(i))
    string = " ".join(L)
    return string

In [45]:
new_data['tags'] = new_data['tags'].apply(stem)
new_data.head(10)

C:\Users\user\AppData\Local\Temp\ipykernel_16132\55449531.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['tags'] = new_data['tags'].apply(stem)


,tconst,primaryTitle,originalTitle,French_Title,tags
0,tt0003883,L'enfant de Paris (1913),L'enfant de Paris (1913),L'Enfant de Paris (1913),crime drama léonceperret louisleuba mauricelag...
1,tt0006206,Les vampires (1915),Les vampires (1915),Les vampires (1915),action adventur crime stacianapierkowska musid...
2,tt0006864,Intolerance (1916),Intolerance: Love's Struggle Throughout the Ag...,Intolérance (1916),drama histori lilliangish robertharron maemars...
3,tt0006886,Judex (1916),Judex (1916),Judex (1916),adventur crime renécresté musidora renépoyen é...
4,tt0009968,Broken Blossoms (1919),Broken Blossoms or The Yellow Man and the Girl...,Le lys brisé (1919),drama romanc normanselbi lilliangish richardba...
5,tt0010307,J'accuse! (1919),J'accuse (1919),J'accuse (1919),drama horror war romualdjoubé maximedesjardin ...
6,tt0011565,The Penalty (1920),The Penalty (1920),Satan (1920),crime drama thriller charlesclari dorispawn ji...
7,tt0011841,Way Down East (1920),Way Down East (1920),À travers l'orage (1920),drama romanc lilliangish richardbarthelmess mr...
8,tt0012364,The Phantom Carriage (1921),Körkarlen (1921),La charrette fantôme (1921),drama fantasi horror einaraxelsson victorsjöst...
9,tt0012494,Destiny (1921),Der müde Tod (1921),Les Trois Lumières (1921),drama fantasi horror bernhardgoetzk lildagov w...


In [46]:
new_data.iloc[8]['tags']

'drama fantasi horror einaraxelsson victorsjöström hildaborgström toresvennberg astridholm concordiaseland lisalundholm torweijden 107min 1921'

In [47]:
#2nd methode de racinisation
#import nltk
#from nltk.stem import PorterStemmer
#ps = PorterStemmer()
#def stems(text):
#    T = []
    
#    for i in text.split():
#        T.append(ps.stem(i))
    
#    return " ".join(T)

#  Construction du modèle

In [48]:
#Création du modèle
model2 = CountVectorizer(max_features=30000, stop_words='english')
#Entrainer le modèle
vector_matrix = model2.fit_transform(new_data['tags'])
vector_matrix.shape

(12322, 23450)

In [49]:
# Cette ligne affiche le vocabulaire avec les indices correspondants après la vectorisation
print(model2.vocabulary_)

{'crime': 4561, 'drama': 5563, 'léonceperret': 12886, 'louisleuba': 12648, 'mauricelagrené': 14029, 'émilekeppen': 23408, 'marcgérard': 13326, 'henriduv': 8210, 'mariedor': 13519, '124min': 24, '1913': 92, 'action': 478, 'adventur': 623, 'stacianapierkowska': 20253, 'musidora': 15152, 'édouardmathé': 23403, 'marcellévesqu': 13318, 'jeanaymé': 9612, 'fernandherrmann': 6567, '421min': 291, '1915': 93, 'histori': 8380, 'lilliangish': 12417, 'robertharron': 18126, 'maemarsh': 12967, 'turn': 21629, '163min': 64, '1916': 94, 'renécresté': 17837, 'renépoyen': 17848, '300min': 273, 'romanc': 18321, 'normanselbi': 16001, 'richardbarthelmess': 17915, 'donaldcrisp': 5431, 'arthurhoward': 2299, 'edwardpeilsr': 5722, 'georgeberang': 7165, '90min': 315, '1919': 95, 'horror': 8442, 'war': 22439, 'romualdjoubé': 18334, 'maximedesjardin': 14055, 'séverin': 20760, 'mar': 13273, 'angèleguy': 1712, '166min': 67, 'thriller': 21237, 'charlesclari': 3942, 'dorispawn': 5502, 'jimmason': 9909, 'lonchaney': 125

In [50]:
#Distance d'un film avec tous les autres films
similarity = cosine_similarity(vector_matrix)

In [51]:
similarity.shape

(12322, 12322)

In [52]:
#Première ligne de la matrice de similarité, qui représente la similarité cosinus entre le premier film et tous les autres films
similarity[0]

array([1.        , 0.09090909, 0.10660036, ..., 0.15075567, 0.10660036,
       0.15075567])

In [53]:
#Matrice similarity
similarity

array([[1.        , 0.09090909, 0.10660036, ..., 0.15075567, 0.10660036,
        0.15075567],
       [0.09090909, 1.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.10660036, 0.        , 1.        , ..., 0.1767767 , 0.125     ,
        0.1767767 ],
       ...,
       [0.15075567, 0.        , 0.1767767 , ..., 1.        , 0.1767767 ,
        0.75      ],
       [0.10660036, 0.        , 0.125     , ..., 0.1767767 , 1.        ,
        0.1767767 ],
       [0.15075567, 0.        , 0.1767767 , ..., 0.75      , 0.1767767 ,
        1.        ]])

# Fonction de recommandation

In [65]:
def recommandation(movie):
    # Votre code pour obtenir les recommandations de films
    # Rechercher le film en fonction de primaryTitle, originalTitle ou frenchTitle
    movie_index = new_data[(new_data['primaryTitle'] == movie) | 
                           (new_data['originalTitle'] == movie) | 
                           (new_data['French_Title'] == movie)].index[0]
    # Extraire les distances du film donné avec tous les autres films
    distances = similarity[movie_index]
    # Trouver les 5 films les plus similaires excepté du film donné 
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    # Créer une liste pour stocker les données des films recommandés
    recommendations_data = []
    
    for i in movies_list:
        title = new_data.iloc[i[0]]['primaryTitle']
        distance = i[1]
        tconst = new_data.iloc[i[0]]['tconst']
        
        # Ajouter les données du film recommandé à la liste
        recommendations_data.append({'tconst': tconst,'Film': title, 'Similarity': distance})
    
    # Créer un DataFrame à partir de la liste des données des films recommandés
    recommendations_df = pd.DataFrame(recommendations_data)
    
    return recommendations_df

In [66]:
# Appel de la fonction recommandation avec un exemple de film
movie_recs_df = recommandation("X-Men (2000)")
# Affichage du DataFrame contenant les recommandations de films
print(movie_recs_df)

      tconst                               Film  Similarity
0  tt1877832  X-Men: Days of Future Past (2014)         0.7
1  tt0290334                          X2 (2003)         0.6
2  tt3315342                       Logan (2017)         0.5
3  tt0078346                    Superman (1978)         0.4
4  tt0082694            The Road Warrior (1981)         0.4


In [70]:
movie_recs_df['tconst']

0    tt1877832
1    tt0290334
2    tt3315342
3    tt0078346
4    tt0082694
Name: tconst, dtype: object